In [1]:
# =========================
# Core
# =========================
import os
import torch
import numpy as np
import pandas as pd

# =========================
# HuggingFace - Models & Tokenizer
# =========================
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling , 
    BitsAndBytesConfig
)


from datasets import load_dataset 
# =========================
# PEFT (LoRA / QLoRA)
# =========================
from peft import (
    LoraConfig,
    get_peft_model,
    TaskType , prepare_model_for_kbit_training
)



c:\Users\10\anaconda3\envs\llm_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
path = r"C:\Users\10\Projects\RNN\cheatsheets"
path_file = []
for file in os.listdir(path):
    path_file.append(os.path.join(path , file))

In [3]:
device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

In [4]:
from torch.utils.data import Dataset , DataLoader
from torch.utils.data import Dataset
import torch

class SlidingWindowDataset(Dataset):
    def __init__(self, files, tokenizer, max_len=512, stride=256):
        self.files = files
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.stride = stride  

        self.tokenizer.pad_token = self.tokenizer.eos_token

        self.samples = []
        self._build_samples()

    def _build_samples(self):
        for file in self.files:
            with open(file, "r", encoding="utf-8") as f:
                text = f.read()

            tokens = self.tokenizer.encode(text)

            # Sliding window
            for i in range(0, len(tokens), self.stride):
                chunk = tokens[i:i + self.max_len]

                if len(chunk) < 10:
                    continue

                self.samples.append(chunk)

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        chunk = self.samples[idx]

        # padding
        input_ids = chunk + [self.tokenizer.pad_token_id] * (self.max_len - len(chunk))

        attention_mask = [1] * len(chunk) + [0] * (self.max_len - len(chunk))

        return {
            "input_ids": torch.tensor(input_ids),
            "attention_mask": torch.tensor(attention_mask),
            "labels": torch.tensor(input_ids)
        }


In [5]:
model_name = "Qwen/Qwen2.5-3B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

In [6]:
dataset = SlidingWindowDataset(path_file , tokenizer)

In [7]:
dataloader = DataLoader(dataset , batch_size=2)

In [8]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)


In [9]:
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
    cache_dir=r"D:\huggingface",
)
model.config.pad_token_id = tokenizer.pad_token_id

model = prepare_model_for_kbit_training(model)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 434/434 [01:56<00:00,  3.72it/s]


In [10]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
    ],
)

model = get_peft_model(model, lora_config)

model.print_trainable_parameters()

trainable params: 7,372,800 || all params: 3,093,311,488 || trainable%: 0.2383


In [11]:
optimizer = torch.optim.AdamW(model.parameters() , lr=2e-4)


In [13]:
from tqdm import tqdm
epochs = 1 
model.train()

step = 0

for epoch in range(epochs):

    loop = tqdm(dataloader, desc=f"Epoch {epoch}")

    for data in loop:

        step += 1

        # move batch to GPU
        input_ids = data["input_ids"].to(device)
        attention_mask = data["attention_mask"].to(device)
        labels = data["labels"].to(device)

        # forward
        out = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = out.loss

        # backward
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # tqdm update
        loop.set_postfix(loss=loss.item(), step=step)

        # print every 10 steps
        if step % 10 == 0:
            print(f"epoch: {epoch} | step: {step} | loss: {loss.item():.4f}")

Epoch 0:   0%|          | 0/942 [00:00<?, ?it/s]

Epoch 0:   1%|          | 10/942 [00:17<27:19,  1.76s/it, loss=1.95, step=10]

epoch: 0 | step: 10 | loss: 1.9535


Epoch 0:   2%|▏         | 20/942 [00:35<26:36,  1.73s/it, loss=0.795, step=20]

epoch: 0 | step: 20 | loss: 0.7948


...

Epoch 0:  99%|█████████▊| 930/942 [27:16<00:20,  1.74s/it, loss=2.33, step=930]

epoch: 0 | step: 930 | loss: 2.3349


Epoch 0: 100%|█████████▉| 940/942 [27:33<00:03,  1.74s/it, loss=1.79, step=940] 

epoch: 0 | step: 940 | loss: 1.7852


Epoch 0: 100%|██████████| 942/942 [27:37<00:00,  1.76s/it, loss=0.775, step=942]


In [25]:
text = "what's xxe"

prompt = "i want payload rce"


In [27]:
inputs = tokenizer(text, return_tensors="pt").to(device)

model.eval()
with torch.no_grad():
    gen = model.generate(**inputs, max_new_tokens=100)

print(tokenizer.decode(gen[0], skip_special_tokens=True))

what's xxe?
XML External Entity (XXE) is a security vulnerability in XML parsers that can be exploited to read files outside of the web application, execute remote code, or perform other malicious actions.
The XXE attack vector is as follows:
An attacker crafts an XML document that includes external entities like
"http://example.com/evil.txt"
or
"file:///etc/passwd"
. When the web application parses this XML document using an XML parser, it will try to resolve the external entity and access


In [28]:
model.save_pretrained(r"D:\my_lora_model")
tokenizer.save_pretrained(r"D:\my_lora_model")

('D:\\my_lora_model\\tokenizer_config.json',
 'D:\\my_lora_model\\chat_template.jinja',
 'D:\\my_lora_model\\tokenizer.json')